In [1]:
import pandas as pd

In [2]:
# Reading data
df = pd.read_csv(r"C:\Users\dhrub\Desktop\Dengue_summary_report\merged_output1.csv")

In [3]:
df

,Disease,Event date,Ward Number,District,Municipality,Hospital,Disease Name
0,Dengue,9/18/2025,10.0,Kaski,NaN,Regional1 hos,NaN
1,Dengue,9/10/2025,8.0,Kaski,Pokhara Metropolitan City,Metrocity1 hos,NaN
2,Dengue,9/13/2025,NaN,Kaski,Pokhara Metropolitan City,Regional1 hos,NaN
3,Dengue,9/8/2025,6.0,Kaski,Pokhara Metropolitan City,Charak1 hos,NaN
4,Dengue,9/8/2025,6.0,Kaski,Pokhara Metropolitan City,Charak1 hos,NaN
...,...,...,...,...,...,...,...
13822,NaN,6/1/2025 0:00,NaN,kaski,Pokhara Metropolitan City,POKHARA ACADEMY OF HEALTH SCIENCES_KASKI,Dengue
13823,NaN,8/14/2025 0:00,11.0,kaski,Pokhara Metropolitan City,MANIPAL TEACHING HOSPITAL_KASKI,Dengue
13824,NaN,1/15/2025 0:00,17.0,kaski,Pokhara Metropolitan City,MANIPAL TEACHING HOSPITAL_KASKI,Dengue
13825,NaN,9/14/2025 0:00,NaN,kaski,NaN,POKHARA ACADEMY OF HEALTH SCIENCES_KASKI,Dengue


In [6]:
df= df.drop(columns=['Disease', 'Disease Name', 'Hospital', 'District'])

In [7]:
df

,Event date,Ward Number,Municipality
0,9/18/2025,10.0,NaN
1,9/10/2025,8.0,Pokhara Metropolitan City
2,9/13/2025,NaN,Pokhara Metropolitan City
3,9/8/2025,6.0,Pokhara Metropolitan City
4,9/8/2025,6.0,Pokhara Metropolitan City
...,...,...,...
13822,6/1/2025 0:00,NaN,Pokhara Metropolitan City
13823,8/14/2025 0:00,11.0,Pokhara Metropolitan City
13824,1/15/2025 0:00,17.0,Pokhara Metropolitan City
13825,9/14/2025 0:00,NaN,NaN


In [8]:
df['Municipality'] = (
    df['Municipality']
    .str.strip()
    .replace({'Pokhara Metropolitan City': 'Pokhara'})
)


In [9]:
df

,Event date,Ward Number,Municipality
0,9/18/2025,10.0,NaN
1,9/10/2025,8.0,Pokhara
2,9/13/2025,NaN,Pokhara
3,9/8/2025,6.0,Pokhara
4,9/8/2025,6.0,Pokhara
...,...,...,...
13822,6/1/2025 0:00,NaN,Pokhara
13823,8/14/2025 0:00,11.0,Pokhara
13824,1/15/2025 0:00,17.0,Pokhara
13825,9/14/2025 0:00,NaN,NaN


In [10]:
print(sorted(df['Municipality'].dropna().unique()))


['11405', '40501', '40502', '40503', '40504', '40505', 'Annapurna Rural Municipality', 'Machhapuchchhre Rural Municipality', 'Madi Rural Municipality', 'Pokhara', 'Rupa Rural Municipality']


In [11]:
df = df[df['Municipality'] == 'Pokhara']


In [12]:
df

,Event date,Ward Number,Municipality
1,9/10/2025,8.0,Pokhara
2,9/13/2025,NaN,Pokhara
3,9/8/2025,6.0,Pokhara
4,9/8/2025,6.0,Pokhara
5,9/4/2025,8.0,Pokhara
...,...,...,...
13821,8/6/2025 0:00,19.0,Pokhara
13822,6/1/2025 0:00,NaN,Pokhara
13823,8/14/2025 0:00,11.0,Pokhara
13824,1/15/2025 0:00,17.0,Pokhara


In [13]:
print(sorted(df['Municipality'].dropna().unique()))


['Pokhara']


In [14]:
# Check missing values
print("Before filling:")
print(df['Ward Number'].isna().sum())

Before filling:
2306


In [15]:
print("Missing ward numbers:", df['Ward Number'].isna().sum())
print("Missing municipality names:", df['Municipality'].isna().sum())


Missing ward numbers: 2306
Missing municipality names: 0


In [16]:
df.duplicated().sum()


6165

In [17]:
df = df.copy()
df['Event date'] = pd.to_datetime(df['Event date'], format='mixed', errors='coerce')


In [31]:
df['year'] = df['Event date'].dt.year
df['month'] = df['Event date'].dt.month


In [32]:
df['Event date'].dtype


dtype('<M8[ns]')

In [33]:
df['Municipality'].unique()


array(['Pokhara'], dtype=object)

In [34]:
df = df.dropna(subset=['Municipality', 'Ward Number'])


In [35]:
# Ensure ward_number is integer
df['Ward Number'] = df['Ward Number'].astype(int)

In [36]:
# Recreate ward_id correctly
df['Ward_id'] = df['Municipality'].astype(str) + '-' + df['Ward Number'].astype(str)

In [37]:
df['Event date'] = pd.to_datetime(df['Event date'])
df['year'] = df['Event date'].dt.year
df['month'] = df['Event date'].dt.month


In [38]:
# Get full ranges
all_wards = df['Ward_id'].unique()
all_years = df['year'].unique()
all_months = range(1, 13)

# Create complete index
full_index = pd.MultiIndex.from_product(
    [all_wards, all_years, all_months],
    names=['Ward_id', 'year', 'month']
)


In [39]:
monthly_ward_cases = (
    df
    .groupby(['Ward_id', 'year', 'month'])
    .size()
    .reindex(full_index, fill_value=0)
    .reset_index(name='dengue_cases')
)


In [40]:
monthly_ward_cases = monthly_ward_cases[
    ['year', 'month', 'dengue_cases', 'Ward_id']
]

monthly_ward_cases.to_csv(
    'monthly_dengue_cases_per_ward_filled.csv',
    index=False
)

print("CSV with zero-filled months created successfully!")


CSV with zero-filled months created successfully!
